In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd

from config import PROCESSED_DATA_DIR

In [2]:
feature_paths = {
    "icustays": PROCESSED_DATA_DIR / "icustays_features.parquet",
    "chartevents": PROCESSED_DATA_DIR / "chartevents_features.parquet",
    "labevents": PROCESSED_DATA_DIR / "labevents_features.parquet",
    "inputevents": PROCESSED_DATA_DIR / "inputevents_features.parquet",
}

missing_paths = [name for name, path in feature_paths.items() if not path.exists()]

if missing_paths:
    missing_display = "\n".join(f"- {name}: {feature_paths[name]}" for name in missing_paths)
    raise FileNotFoundError(
        "Run the upstream feature engineering notebooks before this merge notebook:\n"
        f"{missing_display}"
    )

feature_paths

{'icustays': PosixPath('/Users/brandonwu/Documents/ICU_LOS_Survival_Analysis/data/processed/icustays_features.parquet'),
 'chartevents': PosixPath('/Users/brandonwu/Documents/ICU_LOS_Survival_Analysis/data/processed/chartevents_features.parquet'),
 'labevents': PosixPath('/Users/brandonwu/Documents/ICU_LOS_Survival_Analysis/data/processed/labevents_features.parquet'),
 'inputevents': PosixPath('/Users/brandonwu/Documents/ICU_LOS_Survival_Analysis/data/processed/inputevents_features.parquet')}

In [3]:
icustays_df = pd.read_parquet(feature_paths["icustays"])
chartevents_df = pd.read_parquet(feature_paths["chartevents"])
labevents_df = pd.read_parquet(feature_paths["labevents"])
inputevents_df = pd.read_parquet(feature_paths["inputevents"])

feature_tables = {
    "icustays": icustays_df,
    "chartevents": chartevents_df,
    "labevents": labevents_df,
    "inputevents": inputevents_df,
}

for name, df in feature_tables.items():
    print(f"{name}: {df.shape}")

icustays: (94444, 27)
chartevents: (94444, 71)
labevents: (94444, 271)
inputevents: (94444, 89)


In [4]:
def validate_feature_table(df, name):
    if "stay_id" not in df.columns:
        raise ValueError(f"{name} is missing stay_id")

    duplicate_count = df["stay_id"].duplicated().sum()
    if duplicate_count > 0:
        raise ValueError(f"{name} has {duplicate_count} duplicate stay_id rows")

    print(f"{name}: {len(df):,} rows, {df.shape[1]:,} columns, 0 duplicate stay_id rows")


for name, df in feature_tables.items():
    validate_feature_table(df, name)

icustays: 94,444 rows, 27 columns, 0 duplicate stay_id rows
chartevents: 94,444 rows, 71 columns, 0 duplicate stay_id rows
labevents: 94,444 rows, 271 columns, 0 duplicate stay_id rows
inputevents: 94,444 rows, 89 columns, 0 duplicate stay_id rows


In [5]:
modeling_df = icustays_df.copy()

modeling_df["duration"] = modeling_df["los"]
modeling_df["event_observed"] = 1

print(modeling_df[["duration", "event_observed"]].describe())
print("Non-positive durations:", (modeling_df["duration"] <= 0).sum())
print("Missing durations:", modeling_df["duration"].isna().sum())

           duration  event_observed
count  94444.000000         94444.0
mean       3.630025             1.0
std        5.402474             0.0
min        0.001250             1.0
25%        1.096212             1.0
50%        1.965648             1.0
75%        3.862575             1.0
max      226.403079             1.0
Non-positive durations: 0
Missing durations: 0


In [6]:
if modeling_df["duration"].isna().any():
    raise ValueError("duration has missing values")

if (modeling_df["duration"] <= 0).any():
    raise ValueError("duration has non-positive values")

if not modeling_df["event_observed"].isin([0, 1]).all():
    raise ValueError("event_observed must be binary")

In [7]:
for name, feature_df in [
    ("chartevents", chartevents_df),
    ("labevents", labevents_df),
    ("inputevents", inputevents_df),
]:
    before_shape = modeling_df.shape
    modeling_df = modeling_df.merge(feature_df, on="stay_id", how="left")
    print(f"Merged {name}: {before_shape} -> {modeling_df.shape}")

print("Duplicate stay_id rows:", modeling_df["stay_id"].duplicated().sum())

Merged chartevents: (94444, 29) -> (94444, 99)
Merged labevents: (94444, 99) -> (94444, 369)
Merged inputevents: (94444, 369) -> (94444, 457)
Duplicate stay_id rows: 0


In [8]:
expected_rows = len(icustays_df)

if len(modeling_df) != expected_rows:
    raise ValueError(f"Merged dataset has {len(modeling_df)} rows; expected {expected_rows}")

if modeling_df["stay_id"].duplicated().any():
    raise ValueError("Merged dataset has duplicate stay_id rows")

print(f"Merged dataset validation passed: {modeling_df.shape}")

Merged dataset validation passed: (94444, 457)


In [9]:
# These feature families represent absence of a measured event/exposure, so missing values after the left merge are true zeroes.
zero_fill_suffixes = (
    "_observed_24h",
    "_used_24h",
    "_count_24h",
    "_event_count_24h",
    "_duration_hours_24h",
    "_total_24h",
)

zero_fill_cols = [
    col for col in modeling_df.columns
    if col.endswith(zero_fill_suffixes)
]

modeling_df[zero_fill_cols] = modeling_df[zero_fill_cols].fillna(0)

binary_cols = [col for col in zero_fill_cols if col.endswith(("_observed_24h", "_used_24h"))]
modeling_df[binary_cols] = modeling_df[binary_cols].astype(int)

print(f"Zero-filled columns: {len(zero_fill_cols)}")
print(f"Binary indicator columns: {len(binary_cols)}")

Zero-filled columns: 156
Binary indicator columns: 46


In [10]:
missing_summary_df = (
    modeling_df
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_summary_df["missing_pct"] = missing_summary_df["missing_count"] / len(modeling_df)
missing_summary_df = missing_summary_df.sort_values("missing_pct", ascending=False)

missing_summary_df.head(30)

,missing_count,missing_pct
albumin_std_24h,89898,0.951866
alkaline_phosphatase_std_24h,82319,0.871617
alt_std_24h,82116,0.869468
bilirubin_total_std_24h,82084,0.869129
ast_std_24h,82008,0.868324
albumin_min_24h,69144,0.732116
albumin_mean_24h,69144,0.732116
albumin_last_24h,69144,0.732116
albumin_median_24h,69144,0.732116
albumin_first_24h,69144,0.732116


In [11]:
identifier_cols = ["subject_id", "hadm_id", "stay_id"]
target_cols = ["duration", "event_observed"]
raw_time_cols = [
    col for col in ["intime", "outtime", "admittime"]
    if col in modeling_df.columns
]
raw_outcome_cols = [col for col in ["los"] if col in modeling_df.columns]

excluded_from_modeling = identifier_cols + target_cols + raw_time_cols + raw_outcome_cols
predictor_cols = [col for col in modeling_df.columns if col not in excluded_from_modeling]

print(f"Total columns: {modeling_df.shape[1]}")
print(f"Predictor columns for notebook 10: {len(predictor_cols)}")
print("Excluded columns:", excluded_from_modeling)

Total columns: 457
Predictor columns for notebook 10: 448
Excluded columns: ['subject_id', 'hadm_id', 'stay_id', 'duration', 'event_observed', 'intime', 'outtime', 'admittime', 'los']


In [12]:
print(modeling_df.dtypes.value_counts())
modeling_df.head()

float64           385
int64              55
object             11
datetime64[us]      3
int32               2
category            1
Name: count, dtype: int64


,subject_id,hadm_id,stay_id,first_careunit,intime,outtime,los,gender,anchor_age,admittime,...,other_medication_mcg_total_24h,other_medication_meq_total_24h,other_medication_mg_total_24h,other_ml_total_24h,other_ounces_total_24h,sedative_analgesic_mcg_total_24h,sedative_analgesic_mg_total_24h,vasopressor_inotrope_mcg_total_24h,vasopressor_inotrope_mg_total_24h,vasopressor_inotrope_units_total_24h
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,2180-07-23 12:35:00,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.00000,0.0
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,F,86,2150-11-02 18:02:00,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,9.34123,0.0
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,F,73,2189-06-27 07:38:00,...,0.0,0.0,40.000001,0.0,0.0,0.0,0.0,0.0,0.00000,0.0
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,F,55,2157-11-18 22:56:00,...,0.0,0.0,10.000000,0.0,0.0,0.0,2.0,0.0,0.00000,0.0
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113,F,55,2157-12-18 16:58:00,...,0.0,0.0,1009.999953,0.0,0.0,0.0,6.0,0.0,0.00000,0.0


In [13]:
modeling_df.to_parquet(
    PROCESSED_DATA_DIR / "modeling_dataset.parquet",
    index=False
)

missing_summary_df.to_csv(
    PROCESSED_DATA_DIR / "modeling_dataset_missingness.csv"
)

pd.Series(predictor_cols, name="predictor_column").to_csv(
    PROCESSED_DATA_DIR / "modeling_predictor_columns.csv",
    index=False
)